# ***OUTLINE***

### 1. Install and Imports
### 2. Configure Study Area + Photon-Input Parameters
### 3. Query ATL03 Photon Clouds with atl03x
### 4. HMann: Detect water surface from the height histogram [PER GT: HAYDEN UPDATE]
### 5. Plot Photon Cloud
### 6. Map Photons Over Basemap
### 7. HMann: Segment Binning
### 8. HMann: Variable Retrieval
### 9. HMann: Map Photon heights over a basemap
### 10. HMann: Segment Mapping

In [1]:
"""


Install and imports


"""
from pyproj import Transformer
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import StrMethodFormatter
import warnings
warnings.filterwarnings("ignore")
import contextily as cx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sliderule import icesat2, sliderule
import geopandas as gpd
import rioxarray as rxr
import rasterio
from rasterio.features import rasterize
import numpy as np
from scipy.stats import skew, kurtosis
import geopandas as gpd
from shapely.geometry import LineString




In [2]:
"""

Configure Study Area + Photon-Input Parameters

"""
# Initialize sliderule
sliderule.init("slideruleearth.io", verbose=True)
pd.set_option("display.max_columns", 80)

from imports import params
aoi, resources, base_parms = params()


In [3]:
"""

Query ATL03 photon clouds with atl03x

"""

photons = sliderule.run("atl03x", base_parms)
print(f"rows: {len(photons):,}")
print(f"columns: {len(photons.columns)}")
display(photons.head())


# add lon/lat columns
def add_lon_lat(gdf):
    """Return a copy with lon/lat columns derived from GeoDataFrame geometry."""
    out = gdf.copy()
    if "geometry" in out.columns:
        out["lon"] = out.geometry.x
        out["lat"] = out.geometry.y
    return out

# df of all photons
photons_xy = add_lon_lat(photons)


request <AppServer.22507> retrieved 167 resources
Starting proxy for atl03x to process 167 resource(s) with 40 thread(s)
request <AppServer.24043> on ATL03_20250921074118_01032906_007_01.h5 generated dataframe [gt1r] with 0 rows and 13 columns
request <AppServer.24043> on ATL03_20250921074118_01032906_007_01.h5 generated dataframe [gt3r] with 0 rows and 13 columns
request <AppServer.24043> on ATL03_20250921074118_01032906_007_01.h5 generated dataframe [gt2r] with 0 rows and 13 columns
request <AppServer.24043> on ATL03_20250921074118_01032906_007_01.h5 generated dataframe [gt1l] with 0 rows and 13 columns
request <AppServer.24043> on ATL03_20250921074118_01032906_007_01.h5 generated dataframe [gt2l] with 0 rows and 13 columns
request <AppServer.24043> on ATL03_20250921074118_01032906_007_01.h5 generated dataframe [gt3l] with 0 rows and 13 columns
Successfully completed processing resource [1 out of 167]: ATL03_20250921074118_01032906_007_01.h5
request <AppServer.22785> on ATL03_2024092

rows: 68,547,885
columns: 18


,region,gt,spacecraft_velocity,solar_elevation,rgt,ph_index,height,pce_mframe_cnt,spot,x_atc,y_atc,srcid,cycle,atl03_cnf,background_rate,segment_id,quality_ph,geometry
time_ns,,,,,,,,,,,,,,,,,,
2024-09-23 01:09:37.184672512,6,60,7112.602539,-25.736555,103,616380,-13.882228,524789690,6,1.620561e+07,-10652.596680,6,25,2,4326.103027,809063,0,POINT (-77.55463 34.75819)
2024-09-23 01:09:37.185172480,6,60,7112.602539,-25.736555,103,616381,-18.879610,524789690,6,1.620561e+07,-10652.684570,6,25,4,4326.103027,809063,0,POINT (-77.55464 34.75816)
2024-09-23 01:09:37.185972736,6,60,7112.602539,-25.736555,103,616382,-18.524364,524789690,6,1.620562e+07,-10652.666992,6,25,4,4326.103027,809063,0,POINT (-77.55464 34.75811)
2024-09-23 01:09:37.186472704,6,60,7112.603027,-25.736601,103,616383,-7.352321,524789690,6,1.620562e+07,-10652.449219,6,25,1,4326.103027,809064,0,POINT (-77.55465 34.75807)
2024-09-23 01:09:37.187172608,6,60,7112.603027,-25.736601,103,616384,-18.219751,524789690,6,1.620562e+07,-10652.644531,6,25,4,4326.103027,809064,0,POINT (-77.55465 34.75803)


In [6]:
"""

Mask data to be only over water

atl03x land mask is not good enough. still has data over land.


"""

# # adjust land mask aoi
# pad = 0.5  # buffer (too big just in case) so photons just outside the AOI edge aren't lost
# aoi_lons = [pt["lon"] for pt in aoi]
# aoi_lats = [pt["lat"] for pt in aoi]
# minx = min(aoi_lons) - pad
# maxx = max(aoi_lons) + pad
# miny = min(aoi_lats) - pad
# maxy = max(aoi_lats) + pad

# # # Load land mask: https://www.arcgis.com/home/item.html?id=595533fecdb0472db4b4b8e3ca8d9e42#overview
land = gpd.read_file("ne_10m_land.shp")
land = land.to_crs(photons_xy.crs)

# define resolution (0.01° ~ 1km, adjust as needed)
res = 0.01

# Determine bounding box for the land mask
    # I could probably just do this for my bbox, but I just copied code from a previous project of mine for this
minx, miny, maxx, maxy = -180, -90, 180, 90

# calc number of raster pixels
width = int((maxx - minx) / res)
height = int((maxy - miny) / res)

# coodrinate transformation from pixel loc <--> lat/lon
transform = rasterio.transform.from_bounds(
    minx, miny, maxx, maxy, width, height
)

# rasterize land polygon
land_shapes = [(geom, 1) for geom in land.geometry]

# put it onto an array/raster (bunch of 0s and 1s showing water or land status per pixel)
land_raster = rasterize(
    land_shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype="uint8"
)

# get photon coordinates
xs = photons_xy.geometry.x.values
ys = photons_xy.geometry.y.values

# convert photon coordinates into raster coordinates
col = ((xs - minx) / res).astype(np.int32)
row = ((maxy - ys) / res).astype(np.int32)

# Checks whether photon locs fall in water/land
valid = (
    (col >= 0) & (col < width) &
    (row >= 0) & (row < height)
)

# array of all land vals
on_land = np.zeros(len(photons_xy), dtype=bool)

# mask where row and column (rasterized lat/lon) are both over land
on_land[valid] = land_raster[row[valid], col[valid]] == 1

# limit photons_xy to be oceans only
photons_xy_ocean = photons_xy.loc[~on_land].copy()

In [7]:
"""
Original logic sent by Kelsey Bisson based off of: https://www.researchgate.net/publication/366433397_Novel_application_of_ICESat-2_ATLAS_data_to_determine_coastal_light_attenuation_as_a_proxy_for_suspended_particulate_matter
Code altered by H. Mann (see below)

Detect water surface from the height histogram PER GT


For a lightweight ATL03-only first pass, estimate the water surface from the 
photon-height histogram. The densest height bin is treated as the candidate 
surface return, then `water_surface` keeps photons within an adaptive vertical 
window around that modal height. This is an exploratory detector: inspect the 
histogram diagnostics and tune the binning/window before using the result 
quantitatively.

If the water-surface plot looks like only one or two points, 
first check the printed counts. A sparse or clustered-looking plot can 
happen even with thousands of selected photons when only a few ICESat-2 
pass/beam segments cross the small AOI and `x_atc` is plotted as an absolute 
along-track coordinate.



HAYDEN UPDATE: this should be done for each ground track, as SSH varies 
throughout the tracks, as different ground tracks are on different days.
"""

water_surface_list = []
below_surface_list = []

segment_length = 1000 

for (rgt_val, cycle_val), gt_photons in photons_xy_ocean.groupby(["rgt", "cycle"]):

    # get rid of -infinity and +infinity height readings
    height_values = gt_photons["height"].replace([np.inf, -np.inf], np.nan).dropna()

    # # skip passes with too few photons to build a meaningful histogram
    # if len(height_values) < 30:
    #     continue

    # build histogram range + bins + counts + edges
    hist_range = tuple(height_values.quantile([0.01, 0.99]))
    hist_bins = int(np.clip(np.sqrt(len(height_values)), 40, 160))
    hist_counts, hist_edges = np.histogram(height_values, bins=hist_bins, range=hist_range)

    # water surface bin + bin centers
    water_surface_bin = int(np.argmax(hist_counts))
    water_surface_center = 0.5 * (hist_edges[water_surface_bin] + hist_edges[water_surface_bin + 1])

    # determine peak count in histogram + bin width
    hist_peak_count = int(hist_counts[water_surface_bin])
    hist_bin_width_m = float(hist_edges[1] - hist_edges[0])

    # use bin width to determine the 'half window' for the water surface
    water_surface_half_window_m = max(0.75, 1.5 * hist_bin_width_m)

    # Form a mask for the water surface
    water_surface_mask = gt_photons["height"].between(
        water_surface_center - water_surface_half_window_m,
        water_surface_center + water_surface_half_window_m,
    )

    # assign along-track segment IDs now, while we still have just this GT's
    # photons. Grouping by rgt+cycle (gt is already fixed by the outer loop)
    # handles repeat satellite passes over the same ground track.
    gt_photons = gt_photons.copy()

    ## yes, this starts from 0 for each rgt/cycle track, but the binning variable code below works around this!
    gt_photons["segment_id"] = (
        gt_photons.groupby(["rgt", "cycle"])["x_atc"]
        .transform(lambda x: ((x - x.min()) // segment_length).astype(int))
    )

    # separate photons by height
    water_surface = gt_photons[water_surface_mask].copy()
    below_surface = gt_photons[gt_photons["height"] < water_surface_center - water_surface_half_window_m].copy()

    # determine water surface height resuidual
    water_surface["height_residual_m"] = water_surface["height"] - water_surface_center
    below_surface["height_residual_m"] = below_surface["height"] - water_surface_center


    # determine water surface fraction:
    water_surface_fraction = len(water_surface) / max(len(gt_photons), 1)

    water_surface_list.append(water_surface)
    below_surface_list.append(below_surface)

# combine all gts back together
water_surface = pd.concat(water_surface_list) if water_surface_list else pd.DataFrame()
below_surface = pd.concat(below_surface_list) if below_surface_list else pd.DataFrame()

In [ ]:
# """ 

# Plot the photon cloud


# """

# def sample_for_plot(df, max_points=120_000, random_state=7):
#     if len(df) <= max_points:
#         return df
#     return df.sample(max_points, random_state=random_state)

# plot_df = sample_for_plot(photons_xy_ocean)

# fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

# map_scatter = axes[0].scatter(
#     plot_df["lon"],
#     plot_df["lat"],
#     c=plot_df.get("atl03_cnf", 0),
#     s=1,
#     cmap="viridis",
#     linewidths=0,
# )


# axes[0].set_title("Photon locations")
# axes[0].set_xlabel("Longitude")
# axes[0].set_ylabel("Latitude")
# fig.colorbar(map_scatter, ax=axes[0], label="ATL03 confidence")

# cloud_scatter = axes[1].scatter(
#     plot_df["x_atc"],
#     plot_df["height"],
#     c=plot_df.get("atl03_cnf", 0),
#     s=1,
#     cmap="viridis",
#     linewidths=0,
# )
# axes[1].set_title("Photon cloud")
# axes[1].set_xlabel("Along-track distance, x_atc (m)")
# axes[1].set_ylabel("Photon height (m)")
# # print(plot_df["x_atc"].min())
# # axes[1].set_xlim([3875000, 3900000])
# fig.colorbar(cloud_scatter, ax=axes[1], label="ATL03 confidence")

# plt.show()

In [ ]:
# """ 


# Map Photons over a basemap


# """

# basemap_df = sample_for_plot(below_surface, max_points=50_000).to_crs(epsg=3857)

# fig, ax = plt.subplots(figsize=(8, 8), constrained_layout=True)
# basemap_df.plot(
#     ax=ax,
#     column="height_residual_m" if "height_residual_m" in basemap_df.columns else None,
#     markersize=1,
#     cmap="viridis",
#     alpha=0.75,
#     legend="height" in basemap_df.columns,
# )

# try:
#     cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, attribution_size=6)
# except Exception as exc:
#     print(f"Basemap tiles were unavailable: {exc}")

# ax.set_title("Core Sound ATL03 photon locations")
# ax.set_axis_off()
# plt.show()

In [8]:
"""" BINNING """



# Variables to look at:
#         - N(z): depth-dependent photon distribution
#             - could simplify to percentile depths:
#                 - D10, D25, D50, D75, D90
        
#         - N_int: integrated SUBSURFACE photons
#         - N_surface: SURFACE photon count
#         - R_sw: ratio of N(subsurface) / N(surface)
#         - z_max: max depth of photon
#         - N(z) histogram:
#             - mean depth
#             - std of depth
#             - skewness
#             - kurtosis
#         - K_d?, lat?, lon?, tod?


# segment_vars = {
#     # "N(z)": [],
#     "D10": [],
#     "D25": [],
#     "D50": [],
#     "D75": [],
#     "D90": [],
#     "N_subsurface": [],
#     "N_surface": [],
#     "z_max": [],
#     "R_sw": [],
#     "mean_depth": [],
#     "std of depth": [],
#     "skewness": [],
#     "kurtosis": []
# }

# segment_vars = pd.DataFrame(segment_vars)


def get_segment_stats(below_photons, surface_photons):
    """Returns variables listed above"""

    # no photons --> no stats :(
    if len(below_photons) == 0:
        return {
            "D10": np.nan,
            "D25": np.nan,
            "D50": np.nan,
            "D75": np.nan,
            "D90": np.nan,
            "N_subsurface": 0,
            "N_surface": len(surface_photons),
            "R_sw": 0 if len(surface_photons) > 0 else np.nan,
            "z_max": np.nan,
            "mean_depth": np.nan,
            "std_of_depth": np.nan,
            "skewness": np.nan,
            "kurtosis": np.nan,
            "geo": None
        }


    # photon_depth = np.asarray(below_photons["height"])
    photon_depth = np.asarray(below_photons["height_residual_m"])


    # inverse since height is inverted
    D90, D75, D50, D25, D10 = np.percentile(photon_depth, [10, 25, 50, 75, 90])

    # get stats
    N_subsurface = len(below_photons)
    N_surface = len(surface_photons)
    z_max = abs(np.min(photon_depth))
    R_sw = N_subsurface/N_surface if N_surface > 0 else np.nan
    mean_depth = np.mean(photon_depth)
    std_of_depth = np.std(photon_depth)

    skewness = skew(photon_depth)
    kurtosis_val = kurtosis(photon_depth)

    # segment_geo
    segment = below_photons.sort_values("x_atc")

    # convert photons geo into a linestring for plotting later
    segment_geo = LineString([
        segment.geometry.iloc[0],
        segment.geometry.iloc[-1]
    ])

    return {
        "D10": D10,
        "D25": D25,
        "D50": D50,
        "D75": D75,
        "D90": D90,

        "N_subsurface": N_subsurface,
        "N_surface": N_surface,

        "R_sw": R_sw,
        "z_max": z_max,

        "mean_depth": mean_depth,
        "std_of_depth": std_of_depth,
        "skewness": skewness,
        "kurtosis": kurtosis_val,
        "geo": segment_geo,
        "time": below_photons.index.mean()
    }


rows = []

# columns to differentiate each individual 1 km segment
group_cols = ["rgt", "cycle", "gt", "segment_id"]
below_groups = below_surface.groupby(group_cols)
surface_groups = water_surface.groupby(group_cols)

rows = []

for key, below_group in below_groups:

    # was running into error here, chat hepled fix. just to make sure that groups exist 
    #   (isn't needed in the latest iteration, but did help as a safeguard once)
    try:
        surface_group = surface_groups.get_group(key)
    except KeyError:
        surface_group = water_surface.iloc[0:0]  

    stats = get_segment_stats(below_group, surface_group)
    rows.append(stats)

segment_vars = pd.DataFrame(rows)
segment_vars = gpd.GeoDataFrame(segment_vars, geometry="geo", crs=below_surface.crs)

# revised mask is done LATER

# segment_vars = segment_vars.loc[
#     (segment_vars["N_surface"] + segment_vars["N_subsurface"]) >= 50
# ]

In [9]:
# # SAVE PROCESSED DATA!!! Uncomment if needed

# photons_xy_ocean.to_parquet("data.RAW_photons_OCEAN_xy")
# segment_vars.to_parquet("data.FIXEDBINS_2ksegment_vars")
segment_vars.to_parquet("data.FIXEDBINS_1ksegment_vars")